In [1]:
import os
from google import genai
from qdrant_client import QdrantClient

# 1. เช็ค Qdrant Client
def check_qdrant_methods():
    print("--- Checking QdrantClient Methods ---")
    # สมมติ URL ตามที่คุณใช้ใน ingest.py
    url = "http://localhost:6333" 
    client = QdrantClient(url=url)
    
    # ดึงรายชื่อเมธอดทั้งหมดออกมา
    methods = [m for m in dir(client) if callable(getattr(client, m)) and not m.startswith("_")]
    
    # เช็คว่ามี .search หรือไม่
    if "search" in methods:
        print("✅ พบเมธอด .search() ใน QdrantClient")
    else:
        print("❌ ไม่พบเมธอด .search()! ลองเช็คชื่อเมธอดด้านล่าง:")
        
    print(f"รายการเมธอดทั้งหมด: {', '.join(methods)}")

# 2. เช็ค Gemini Client (เผื่อไว้)
def check_gemini_methods():
    print("\n--- Checking Gemini Client Methods ---")
    api_key = os.environ.get("GEMINI_API_KEY")
    if api_key:
        client = genai.Client(api_key=api_key)
        methods = [m for m in dir(client) if callable(getattr(client, m)) and not m.startswith("_")]
        print(f"รายการเมธอด Gemini Client: {', '.join(methods)}")
    else:
        print("ข้ามการเช็ค Gemini (ไม่ได้ตั้งค่า API Key)")

if __name__ == "__main__":
    check_qdrant_methods()
    check_gemini_methods()

--- Checking QdrantClient Methods ---
❌ ไม่พบเมธอด .search()! ลองเช็คชื่อเมธอดด้านล่าง:
รายการเมธอดทั้งหมด: add, batch_update_points, clear_payload, close, cluster_collection_update, cluster_status, cluster_telemetry, collection_cluster_info, collection_exists, count, create_collection, create_full_snapshot, create_payload_index, create_shard_key, create_shard_snapshot, create_snapshot, create_vector_name, delete, delete_collection, delete_full_snapshot, delete_payload, delete_payload_index, delete_shard_key, delete_shard_snapshot, delete_snapshot, delete_vector_name, delete_vectors, facet, get_aliases, get_collection, get_collection_aliases, get_collections, get_embedding_size, get_fastembed_sparse_vector_params, get_fastembed_vector_params, get_optimizations, get_sparse_vector_field_name, get_vector_field_name, info, list_full_snapshots, list_image_models, list_late_interaction_multimodal_models, list_late_interaction_text_models, list_shard_keys, list_shard_snapshots, list_snapshots

C:\Users\BM MONEY\AppData\Local\Temp\ipykernel_26760\2692702064.py:29: UserWarning: Agents usage is experimental and may change in future versions.
  methods = [m for m in dir(client) if callable(getattr(client, m)) and not m.startswith("_")]
C:\Users\BM MONEY\AppData\Local\Temp\ipykernel_26760\2692702064.py:29: UserWarning: Interactions usage is experimental and may change in future versions.
  methods = [m for m in dir(client) if callable(getattr(client, m)) and not m.startswith("_")]


In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

# Local Qdrant (ไม่ต้องเปิด Docker)
client = QdrantClient(":memory:")

# Create Collection
client.create_collection(
    collection_name="test_docs",
    vectors_config=VectorParams(
        size=4,
        distance=Distance.COSINE
    )
)

# Insert vectors
client.upsert(
    collection_name="test_docs",
    points=[
        PointStruct(
            id=1,
            vector=[0.9, 0.1, 0.1, 0.1],
            payload={"text": "Diabetes Guideline"}
        ),
        PointStruct(
            id=2,
            vector=[0.8, 0.2, 0.1, 0.1],
            payload={"text": "Hypertension Guideline"}
        ),
        PointStruct(
            id=3,
            vector=[0.1, 0.9, 0.1, 0.1],
            payload={"text": "Pneumonia Guideline"}
        )
    ]
)

print("Insert Success")

Insert Success


In [4]:
result = client.query_points(
    collection_name="test_docs",
    query=[0.85, 0.15, 0.1, 0.1],
    limit=2
)

for point in result.points:
    print(
        point.id,
        point.score,
        point.payload
    )

1 0.9979749155678157 {'text': 'Diabetes Guideline'}
2 0.9975694184303856 {'text': 'Hypertension Guideline'}


In [7]:
from qdrant_client.http import models

print(hasattr(models, "QueryVector"))

False
